# Semaine 2 — Recherche par le sens : embeddings + base vectorielle

**Objectif (énoncé de l'enseignant)** : découper les documents, calculer les embeddings, monter la base vectorielle, puis comparer recherche par mots-clés et recherche sémantique sur nos questions.

| Étape | Où |
|---|---|
| 1. Découper les PDF en passages (chunks) | `src/build_chunks.py` → `data/processed/chunks.jsonl` |
| 2. Calculer les embeddings + monter la base vectorielle | **ce notebook** |
| 3. Comparer mots-clés / sémantique sur les questions | `python src/compare_search.py` → `docs/evaluation/comparaison_recherche.md` |

Les embeddings sont calculés **localement** (modèle `sentence-transformers`), Groq n'est pas utilisé ici : il ne servira qu'à rédiger la réponse finale.

## 0. Préparation

Sur **Google Colab** : importez le dossier `COGNIVEX` (zip), puis décommentez les 2 lignes ci-dessous.
En local : `pip install -r requirements.txt` une seule fois, depuis la racine du projet.

In [1]:
import sys
from pathlib import Path

# On remonte jusqu'à trouver src/rag_config.py (marche depuis COGNIVEX/ ou COGNIVEX/notebooks/)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "rag_config.py").exists()), None)
if ROOT is None:
    # Recherche locale dans le dossier extrait si le CWD ne permet pas de remonter directement
    config_paths = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        try:
            config_paths.extend(base.rglob("src/rag_config.py"))
        except PermissionError:
            pass

    if config_paths:
        ROOT = config_paths[0].parent.parent

    if ROOT is None:
        raise FileNotFoundError(
            "Extrayez puis ouvrez le notebook depuis le dossier COGNIVEX "
            "(C:\\...\\COGNIVEX\\notebooks\\recherche.ipynb)."
        )
sys.path.insert(0, str(ROOT / "src"))
print("Racine du projet :", ROOT)

from rag_config import (COLLECTION_NAME, EMBEDDING_MODEL, TOP_K, VECTOR_DB_DIR,
                        charger_chunks, passage_prefix, query_prefix, texte_indexe)
print("Modèle d'embeddings :", EMBEDDING_MODEL)

Racine du projet : c:\Users\Hp\Desktop\COGNIVEX
Modèle d'embeddings : intfloat/multilingual-e5-small


## 1. Charger les chunks

`charger_chunks()` lit `data/processed/chunks.jsonl` (produit par `build_chunks.py`), retire le bruit d'impression du navigateur (dates, URL de pied de page) et donne un identifiant unique à chaque chunk.

In [3]:
import json
from collections import Counter
from rag_config import (
    CHUNKS_FILE,
    MIN_CHUNK_CHARS,
    NETTOYER_BRUIT,
    charger_chunks,
    nettoyer_texte,
)

with CHUNKS_FILE.open(encoding="utf-8") as fichier:
    chunks_bruts = [json.loads(ligne) for ligne in fichier if ligne.strip()]

chunks = charger_chunks()

bruts_par_dci = Counter(chunk["dci"] for chunk in chunks_bruts)
gardes_par_dci = Counter(chunk["dci"] for chunk in chunks)

textes_modifies = 0
for chunk in chunks_bruts:
    texte = chunk["text"]
    texte_nettoye = nettoyer_texte(texte) if NETTOYER_BRUIT else texte

    if texte_nettoye != texte:
        textes_modifies += 1

print(f"Chunks avant nettoyage : {len(chunks_bruts)}")
print(f"Chunks conservés après nettoyage : {len(chunks)}")
print(f"Chunks éliminés : {len(chunks_bruts) - len(chunks)}")
print(f"Chunks dont le texte a été modifié : {textes_modifies}\n")

print("Détail par DCI :")
for dci in sorted(bruts_par_dci):
    avant = bruts_par_dci[dci]
    apres = gardes_par_dci[dci]
    elimines = avant - apres
    print(f"{dci:40s} avant : {avant:5d} | après : {apres:5d} | éliminés : {elimines:5d}")

Chunks avant nettoyage : 3993
Chunks conservés après nettoyage : 3989
Chunks éliminés : 4
Chunks dont le texte a été modifié : 996

Détail par DCI :
ACIDE-ACETYLSALICYLIQUE                  avant :    57 | après :    57 | éliminés :     0
AMLODIPINE                               avant :   211 | après :   211 | éliminés :     0
AMOXICILLINE                             avant :   551 | après :   550 | éliminés :     1
AMOXICILLINE + ACIDE CLAVULANIQUE        avant :   207 | après :   204 | éliminés :     3
ATÉNOLOL                                 avant :    64 | après :    64 | éliminés :     0
BISOPROLOL                               avant :    63 | après :    63 | éliminés :     0
CLINDAMYCINE                             avant :    82 | après :    82 | éliminés :     0
CÉFIXIME                                 avant :   403 | après :   403 | éliminés :     0
CÉFUROXIME                               avant :   199 | après :   199 | éliminés :     0
CÉLÉCOXIB                                

## 2. Calculer les embeddings

Un **embedding** transforme un texte en une liste de nombres (ici 384) : deux textes de sens proche donnent des vecteurs proches. Le modèle `multilingual-e5-small` comprend le français et accepte 512 tokens (nos chunks de 120 mots passent sans être tronqués). Il demande le préfixe `passage: ` côté documents et `query: ` côté questions.

> La 1re exécution télécharge le modèle (~470 Mo, internet nécessaire).

In [5]:
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer(EMBEDDING_MODEL)
print("Modèle chargé :", modele is not None)
print("Dimension :", modele.get_embedding_dimension())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 817.97it/s]


Modèle chargé : True
Dimension : 384


In [7]:
import numpy as np
import time
from rag_config import passage_prefix, texte_indexe

textes = [passage_prefix() + texte_indexe(chunk) for chunk in chunks]

debut = time.time()
embeddings = modele.encode(
    textes,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

fichier = ROOT / "data" / "processed" / "embeddings" / "embeddings.npy"
fichier.parent.mkdir(parents=True, exist_ok=True)
np.save(fichier, embeddings)

print(f"{len(embeddings)} embeddings calculés en {time.time() - debut:.0f} secondes")
print("Dimension :", embeddings.shape[1])
print("Fichier enregistré :", fichier)

Batches: 100%|██████████| 125/125 [20:56<00:00, 10.05s/it]

3989 embeddings calculés en 1256 secondes
Dimension : 384
Fichier enregistré : c:\Users\Hp\Desktop\COGNIVEX\data\processed\embeddings\embeddings.npy


In [8]:
print("Norme du 1er vecteur :", round(float(np.linalg.norm(embeddings[0])), 4), "(1.0 = normalisé)")
print("8 premières valeurs  :", np.round(embeddings[0][:8], 3))

Norme du 1er vecteur : 1.0 (1.0 = normalisé)
8 premières valeurs  : [ 0.011 -0.024 -0.072 -0.052  0.046 -0.035  0.03   0.101]


## 3. Monter la base vectorielle (ChromaDB)

On stocke pour chaque chunk : son vecteur, son texte brut et ses métadonnées (DCI, produit, type de document, fichier, page, section) — de quoi **citer la source** (fichier + page) dans la réponse finale. La base est reconstruite à chaque exécution : pas de doublons.

In [10]:
import chromadb

from rag_config import COLLECTION_NAME, VECTOR_DB_DIR

client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)

print("Vecteurs dans ChromaDB :", collection.count())

Vecteurs dans ChromaDB : 3989


## 4. Test rapide : recherche par le sens

`chercher()` transforme la question en vecteur (préfixe `query: `) et retourne les chunks les plus proches. Les questions viennent de `docs/evaluation/questions_llm.md`.

In [11]:
def chercher(question, k=TOP_K):
    emb = modele.encode([query_prefix() + question], normalize_embeddings=True)
    res = collection.query(query_embeddings=emb.tolist(), n_results=k)
    for rang, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), 1):
        print(f"{rang}. similarité {1 - dist:.3f} | {meta['dci']} | {meta['produit'] or '-'} | "
              f"{meta['doc_type']} p.{meta['page']}")
        print("   ", " ".join(doc.split())[:200], "\n")


from compare_search import charger_questions

for q in charger_questions()[:3]:          # changez [:3] pour tester d'autres questions
    print("=" * 100)
    print(q["id"], "—", q["texte"], "\n")
    chercher(q["texte"], k=3)

Q1 — Quelle est la liste des formes pharmaceutiques et des dosages de l’association amoxicilline + acide clavulanique répertoriés dans la nomenclature pharmaceutique algérienne consultée ? 

1. similarité 0.898 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.18
    Patients insuffisants hépatiques L’association amoxicilline/acide clavulanique doit être utilisée avec prudence chez les patients insuffisants hépatiques et la fonction hépatique doit être surveillée  

2. similarité 0.897 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.2
    concernant l’utilisation appropriée des antibactériens. 4.2. Posologie et mode d'administration Posologie Les doses sont exprimées en quantité d’amoxicilline/acide clavulanique, sauf lorsqu’elles sont 

3. similarité 0.896 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.18
    à des hommes et des femmes sains, le sexe n’a pas d’incidence significa

## 5. Étape suivante : comparer avec la recherche par mots-clés

Depuis un terminal, à la racine du projet :

```
python src/compare_search.py
```

Le script pose **toutes les questions** de `docs/evaluation/questions_llm.md` aux deux méthodes (BM25 par mots-clés et recherche sémantique), sans réponse de référence, et écrit le rapport `docs/evaluation/comparaison_recherche.md` (extraits retrouvés, médicaments retrouvés, chunks communs aux deux méthodes).